##leccion 2

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/la_liga.csv",
    encoding="utf-16"
)

df.head()

,Season,Game,Score,Teams,Home Team,Away Team,Home Goals,Away Goals,Home Points,Away Points,Pools
0,1929,1,2-3,Arenas de Getxo - Atlético,Arenas de Getxo,Atlético,2,3,0,2,2
1,1929,1,3-2,Espanyol - Real Unión de Irún,Espanyol,Real Unión de Irún,3,2,2,0,1
2,1929,1,1-1,R. Sociedad - Athletic,R. Sociedad,Athletic,1,1,1,1,X
3,1929,1,5-0,Real Madrid - CE Europa,Real Madrid,CE Europa,5,0,2,0,1
4,1929,1,0-2,Racing - Barcelona,Racing,Barcelona,0,2,0,2,2


In [2]:
print(df.shape)

df.info()

df.isna().sum()

(26558, 11)
<class 'pandas.DataFrame'>
RangeIndex: 26558 entries, 0 to 26557
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Season       26558 non-null  int64
 1   Game         26558 non-null  int64
 2   Score        26558 non-null  str  
 3   Teams        26558 non-null  str  
 4   Home Team    26558 non-null  str  
 5   Away Team    26558 non-null  str  
 6   Home Goals   26558 non-null  int64
 7   Away Goals   26558 non-null  int64
 8   Home Points  26558 non-null  int64
 9   Away Points  26558 non-null  int64
 10  Pools        26558 non-null  str  
dtypes: int64(6), str(5)
memory usage: 2.2 MB


Season         0
Game           0
Score          0
Teams          0
Home Team      0
Away Team      0
Home Goals     0
Away Goals     0
Home Points    0
Away Points    0
Pools          0
dtype: int64

In [3]:
df.describe()

,Season,Game,Home Goals,Away Goals,Home Points,Away Points
count,26558.000000,26558.000000,26558.000000,26558.000000,26558.000000,26558.000000
mean,1985.639883,17.760863,1.797575,1.038068,1.522404,0.780443
std,24.388241,10.382684,1.485426,1.083523,1.032444,1.021123
min,1929.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,1967.000000,9.000000,1.000000,0.000000,1.000000,0.000000
50%,1989.000000,17.000000,2.000000,1.000000,2.000000,0.000000
75%,2006.000000,26.000000,3.000000,2.000000,2.000000,1.000000
max,2023.000000,42.000000,12.000000,8.000000,3.000000,3.000000


Lección 2: Limpieza y estandarización de datos

In [4]:
duplicados = df.duplicated().sum()

print(f"Duplicados encontrados: {duplicados}")

Duplicados encontrados: 0


Se verificó la existencia de registros duplicados en el conjunto de datos. El resultado obtenido fue de 0 registros duplicados, por lo que no fue necesario aplicar procesos de eliminación mediante drop_duplicates().

In [5]:
df.isna().sum()

Season         0
Game           0
Score          0
Teams          0
Home Team      0
Away Team      0
Home Goals     0
Away Goals     0
Home Points    0
Away Points    0
Pools          0
dtype: int64

Se analizaron los valores faltantes en todas las variables del dataset. No se identificaron valores nulos, por lo que no fue necesario aplicar técnicas de imputación o eliminación de registros.

In [6]:
df_clean = df.copy()

df_clean["Home Team"] = (
    df_clean["Home Team"]
    .str.strip()
)

df_clean["Away Team"] = (
    df_clean["Away Team"]
    .str.strip()
)

In [7]:
print(df_clean["Home Team"].head())
print(df_clean["Away Team"].head())

0    Arenas de Getxo
1           Espanyol
2        R. Sociedad
3        Real Madrid
4             Racing
Name: Home Team, dtype: str
0              Atlético
1    Real Unión de Irún
2              Athletic
3             CE Europa
4             Barcelona
Name: Away Team, dtype: str


Se aplicó la función str.strip() para eliminar posibles espacios en blanco al inicio o al final de los nombres de los equipos. Esta acción contribuye a mantener la consistencia de la información durante los procesos de agrupación y análisis.

In [8]:
def clean_data(df):

    df = df.copy()

    df = df.drop_duplicates()

    df["Home Team"] = df["Home Team"].str.strip()
    df["Away Team"] = df["Away Team"].str.strip()

    return df

In [9]:
df_clean = (
    df
    .pipe(clean_data)
)

df_clean.head()

,Season,Game,Score,Teams,Home Team,Away Team,Home Goals,Away Goals,Home Points,Away Points,Pools
0,1929,1,2-3,Arenas de Getxo - Atlético,Arenas de Getxo,Atlético,2,3,0,2,2
1,1929,1,3-2,Espanyol - Real Unión de Irún,Espanyol,Real Unión de Irún,3,2,2,0,1
2,1929,1,1-1,R. Sociedad - Athletic,R. Sociedad,Athletic,1,1,1,1,X
3,1929,1,5-0,Real Madrid - CE Europa,Real Madrid,CE Europa,5,0,2,0,1
4,1929,1,0-2,Racing - Barcelona,Racing,Barcelona,0,2,0,2,2


Se implementó un pipeline utilizando pipe() con el fin de automatizar y estandarizar el proceso de limpieza del dataset.

In [10]:
assert df_clean.duplicated().sum() == 0

assert df_clean["Home Team"].notna().all()

assert df_clean["Away Team"].notna().all()

print("Validación superada")

Validación superada


Las validaciones realizadas confirmaron que el dataset no presenta registros duplicados ni valores faltantes en las variables de equipos, garantizando la calidad de los datos para las siguientes etapas del proyecto.

# Lección 3: Filtrado, ordenamiento y agrupación de datos

En esta sección se aplican técnicas de análisis de datos para identificar patrones de rendimiento en los equipos de LaLiga. Para ello se utilizan operaciones de filtrado, agrupación y cálculo de indicadores que permitan responder la pregunta analítica planteada.

**Pregunta analítica:**

¿Cómo influyen los goles anotados y los puntos obtenidos en el rendimiento histórico de los equipos de LaLiga?

In [11]:
goles_temporada = (
    df_clean
    .groupby("Season")[["Home Goals", "Away Goals"]]
    .mean()
    .round(2)
)

goles_temporada.head(10)

,Home Goals,Away Goals
Season,,
1929,2.73,1.52
1930,3.03,1.63
1931,2.74,1.52
1932,2.63,1.27
1933,2.92,1.52
1934,3.01,1.19
1935,2.64,1.46
1936,2.72,1.28
1940,2.48,1.32


Este indicador permite identificar el comportamiento ofensivo de LaLiga a lo largo de las temporadas. Los resultados muestran el promedio de goles anotados por los equipos locales y visitantes en cada temporada.

In [12]:
goles_local = (
    df_clean
    .groupby("Home Team")["Home Goals"]
    .sum()
)

goles_visitante = (
    df_clean
    .groupby("Away Team")["Away Goals"]
    .sum()
)

total_goles = (
    goles_local
    .add(goles_visitante, fill_value=0)
    .sort_values(ascending=False)
)

total_goles.head(10)

Home Team
Barcelona        6395
Real Madrid      6391
Atlético         4894
Athletic         4888
Valencia         4700
Sevilla          3995
Espanyol         3805
R. Sociedad      3451
Real Zaragoza    2680
Celta            2568
dtype: int64

Este indicador muestra los equipos con mayor capacidad ofensiva en la historia de LaLiga. La suma de goles como local y visitante permite obtener una visión completa del rendimiento goleador de cada club.

In [13]:
puntos_local = (
    df_clean
    .groupby("Home Team")["Home Points"]
    .sum()
)

puntos_visitante = (
    df_clean
    .groupby("Away Team")["Away Points"]
    .sum()
)

total_puntos = (
    puntos_local
    .add(puntos_visitante, fill_value=0)
    .sort_values(ascending=False)
)

total_puntos.head(10)

Home Team
Real Madrid      4861
Barcelona        4762
Atlético         3899
Valencia         3706
Athletic         3665
Sevilla          3201
Espanyol         2995
R. Sociedad      2788
Real Betis       2219
Real Zaragoza    2109
dtype: int64

El total de puntos acumulados representa una medida directa del rendimiento histórico de los equipos. Los clubes ubicados en los primeros lugares corresponden a aquellos que han mantenido mayor regularidad competitiva a lo largo de las temporadas.

In [14]:
temporadas_modernas = df_clean.query("Season >= 2000")

temporadas_modernas.head()

,Season,Game,Score,Teams,Home Team,Away Team,Home Goals,Away Goals,Home Points,Away Points,Pools
17438,2000,1,1-2,Mallorca - Real Madrid,Mallorca,Real Madrid,1,2,0,3,2
17439,2000,1,1-2,Valencia - Racing,Valencia,Racing,1,2,0,3,2
17440,2000,1,1-0,Athletic - Real Betis,Athletic,Real Betis,1,0,3,0,1
17441,2000,1,0-2,Atlético - Rayo Vallecano,Atlético,Rayo Vallecano,0,2,0,3,2
17442,2000,1,1-0,Numancia - Real Valladolid,Numancia,Real Valladolid,1,0,3,0,1


Se filtraron las temporadas desde el año 2000 para analizar el comportamiento reciente del campeonato y evitar sesgos asociados a diferencias históricas en las reglas de competencia.

In [15]:
partidos_mas_goles = (
    df_clean
    .assign(
        Total_Goals=df_clean["Home Goals"] + df_clean["Away Goals"]
    )
    .sort_values("Total_Goals", ascending=False)
)

partidos_mas_goles[
    ["Season", "Teams", "Home Goals", "Away Goals", "Total_Goals"]
].head(10)

,Season,Teams,Home Goals,Away Goals,Total_Goals
410,1933,Athletic - Racing,9,5,14
953,1941,Sevilla - Valencia,10,3,13
2739,1951,Athletic - Celta,9,4,13
5031,1960,Real Madrid - Elche,11,2,13
225,1931,Athletic - Barcelona,12,1,13
939,1941,Sevilla - Barcelona,11,1,12
2643,1950,Atlético - Athletic,6,6,12
1717,1945,Valencia - Real Murcia,9,3,12
840,1940,Athletic - Barcelona,7,5,12
23672,2016,Real Madrid - Rayo Vallecano,10,2,12


La tabla presenta los partidos con mayor cantidad de goles registrados en el dataset. Esta información permite identificar encuentros atípicos y analizar comportamientos extremos dentro de la competición.

In [16]:
resumen_temporada = (
    df_clean
    .groupby("Season")
    .agg({
        "Home Goals": ["mean", "sum"],
        "Away Goals": ["mean", "sum"]
    })
)

resumen_temporada.head()

Home Goals      Away Goals     
             mean  sum       mean  sum
Season                                
1929     2.733333  246   1.522222  137
1930     3.033333  273   1.633333  147
1931     2.744444  247   1.522222  137
1932     2.633333  237   1.266667  114
1933     2.922222  263   1.522222  137

La función agg() permite obtener múltiples métricas de resumen en una sola operación. En este caso se calculan los promedios y totales de goles anotados por temporada.

## Conclusiones de análisis

Los indicadores calculados muestran que existe una relación directa entre la capacidad goleadora y el rendimiento de los equipos en LaLiga. Los clubes con mayor acumulación histórica de goles también suelen ocupar las posiciones más altas en términos de puntos obtenidos.

Asimismo, el análisis permitió identificar patrones históricos de rendimiento y diferencias entre los resultados obtenidos como local y como visitante. Estos hallazgos constituyen una base sólida para las siguientes etapas del proyecto, donde se profundizará en el análisis y visualización de los datos.

# Lección 4: Estructuración y validación del dataset

El proyecto utiliza un único conjunto de datos consolidado que contiene toda la información necesaria para responder la pregunta analítica planteada. Por esta razón, no fue necesario aplicar operaciones de combinación mediante merge() o concat().

Las actividades de esta etapa se enfocan en validar la integridad de los datos, estructurar el dataset final y prepararlo para las siguientes fases del proyecto.

In [17]:
print("Filas y columnas:")
print(df_clean.shape)

print("\nDuplicados:")
print(df_clean.duplicated().sum())

print("\nValores nulos:")
print(df_clean.isna().sum().sum())

Filas y columnas:
(26558, 11)

Duplicados:
0

Valores nulos:
0


La validación realizada confirma que el dataset no contiene registros duplicados ni valores faltantes. Esto garantiza que la información utilizada en el análisis mantiene consistencia e integridad.

In [18]:
assert (df_clean["Home Goals"] >= 0).all()
assert (df_clean["Away Goals"] >= 0).all()

assert (df_clean["Home Points"] >= 0).all()
assert (df_clean["Away Points"] >= 0).all()

print("Validación de variables numéricas superada.")

Validación de variables numéricas superada.


Se comprobó que las variables numéricas relacionadas con goles y puntos contienen únicamente valores válidos y coherentes para el contexto del fútbol profesional.

In [19]:
assert df_clean["Home Team"].notna().all()
assert df_clean["Away Team"].notna().all()

print("Validación de equipos superada.")

Validación de equipos superada.


Se verificó que todos los registros poseen información de equipo local y visitante, evitando inconsistencias durante los análisis posteriores.

In [20]:
df_final = df_clean.copy()

print(df_final.head())

   Season  Game Score                          Teams        Home Team  \
0    1929     1   2-3     Arenas de Getxo - Atlético  Arenas de Getxo   
1    1929     1   3-2  Espanyol - Real Unión de Irún         Espanyol   
2    1929     1   1-1         R. Sociedad - Athletic      R. Sociedad   
3    1929     1   5-0        Real Madrid - CE Europa      Real Madrid   
4    1929     1   0-2             Racing - Barcelona           Racing   

            Away Team  Home Goals  Away Goals  Home Points  Away Points Pools  
0            Atlético           2           3            0            2     2  
1  Real Unión de Irún           3           2            2            0     1  
2            Athletic           1           1            1            1     X  
3           CE Europa           5           0            2            0     1  
4           Barcelona           0           2            0            2     2  


Después de las tareas de limpieza y validación, se construyó un dataset estructurado y preparado para las siguientes etapas del proyecto de análisis de datos.

In [21]:
df_final.to_csv(
    "../data/processed/laliga_clean.csv",
    index=False
)

print("Dataset exportado correctamente.")

Dataset exportado correctamente.


In [22]:
import os

archivo = "../data/processed/laliga_clean.csv"

print(os.path.exists(archivo))

True


La exportación fue exitosa y el archivo quedó almacenado en la carpeta data/processed, cumpliendo con los requisitos establecidos en la guía.

## Conclusiones de estructuración

Durante esta etapa se validó la calidad e integridad del dataset utilizado en el proyecto. Debido a que el análisis se desarrolla sobre un único conjunto de datos, no fue necesario realizar procesos de combinación mediante merge() o concat().

Las verificaciones realizadas demostraron que el dataset no presenta registros duplicados ni valores faltantes y que las variables numéricas contienen valores consistentes. Finalmente, se generó una versión estructurada del dataset y se exportó correctamente a la carpeta data/processed para su utilización en las siguientes fases del proyecto.